# EDA Part A: Business structure and semantics of the text

**Question:** Can high-risk intents be separated from ordinary customer questions by meaning alone?

Data: the cleaned Banking77 sentences (`data/processed/banking77_clean.parquet`, 13,083 rows, with `split`) and the intent catalog (`data/processed/dim_intent.parquet`). This part does not use the simulated stream. `risk_level` and `fraud_type` are assigned to the intent in `dim_intent`. Every sentence with the same intent receives the same label. These figures describe that taxonomy and the wording of the catalog. They are not measurements of confirmed fraudulent transactions.

Sentence embeddings are computed once with `all-MiniLM-L6-v2` and saved under `eda/output/`.

| Figure | What it answers |
|------------------------------------|------------------------------------|
| A1 | How is assigned risk composed across the 11 journey stages? |
| A2 | Which intents sit inside each stage? |
| A3 | Do high-risk intents occupy their own region of meaning, including after removing neighbors of the same intent? |
| A4 | Does sentence length differ by assigned risk? |

In [1]:
import hashlib
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.neighbors import NearestNeighbors

ROOT = Path.cwd().parent if Path.cwd().name == "eda" else Path.cwd()
FIG_DIR = ROOT / "figures"
OUT_DIR = ROOT / "eda" / "output"
FIG_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

EMB_PATH = OUT_DIR / "A_minilm_embeddings.npy"
UMAP_PATH = OUT_DIR / "A_umap_xy.npy"
META_PATH = OUT_DIR / "A_embedding_meta.json"
MODEL_NAME = "all-MiniLM-L6-v2"

RISK_COLORS = {"low": "#5dade2", "medium": "#f5b041", "high": "#c0392b"}
FRAUD_COLORS = {
    "(ordinary)": "#d5d8dc",
    "account_takeover": "#c0392b",
    "card_compromise": "#e67e22",
    "identity_kyc_bypass": "#8e44ad",
    "app_fraud_money_mule": "#d68910",
    "refund_chargeback_abuse": "#1abc9c",
    "virtual_card_abuse": "#2471a3",
}
FRAUD_LABELS = {
    "(ordinary)": "ordinary",
    "account_takeover": "account takeover",
    "card_compromise": "card compromise",
    "identity_kyc_bypass": "identity / KYC",
    "app_fraud_money_mule": "money mule",
    "refund_chargeback_abuse": "refund abuse",
    "virtual_card_abuse": "virtual card",
}
JOURNEY_ORDER = [
    "card_setup", "card_usage", "payments", "cash", "top_up",
    "transfers", "limits", "account_access", "identity",
    "foreign_exchange", "refunds",
]
# Outer-ring names on the static sunburst. Every intent remains available on hover.
LABEL_INTENTS = {
    "change_pin", "edit_personal_details", "lost_or_stolen_phone", "passcode_forgotten",
    "pin_blocked", "compromised_card", "lost_or_stolen_card",
    "card_payment_not_recognised", "cash_withdrawal_not_recognised",
    "unable_to_verify_identity", "verify_my_identity", "why_verify_identity",
}

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

def save_fig(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", dpi=300, bbox_inches="tight", facecolor="white")
    fig.savefig(FIG_DIR / f"{name}.pdf", bbox_inches="tight", facecolor="white")

clean = pd.read_parquet(ROOT / "data/processed/banking77_clean.parquet")
dim = pd.read_parquet(ROOT / "data/processed/dim_intent.parquet")
df = clean.merge(dim.drop(columns=["intent"]), left_on="label", right_on="intent_id", how="left")
if df["risk_level"].isna().any():
    raise SystemExit("dim_intent does not cover every Banking77 label")
df = df.sort_values(["label", "split", "text"]).reset_index(drop=True)
df["fraud_type_plot"] = df["fraud_type"].fillna("(ordinary)")
print(f"sentences={len(df)} intents={df['intent'].nunique()} risk={df['risk_level'].value_counts().to_dict()}")


sentences=13083 intents=77 risk={'low': 9466, 'high': 1887, 'medium': 1730}

## 1 A1. How is assigned risk composed across journey stages?

Banking77 is a catalog of 77 intents, so a count is a property of the corpus, not of live chat volume. This chart rolls the intent labels up to the 11 journey stages. The share is what matters for the question: a stage with many sentences can still be almost entirely ordinary, and a smaller stage can be mostly high-risk. A1 says how risk is composed. A2 names the intents inside each stage.

In [3]:
intent_counts = (
    df.groupby(["journey_stage", "intent", "risk_level", "fraud_type"], dropna=False)
    .agg(n=("text", "size"), median_chars=("text_length", "median"))
    .reset_index()
)
stage_rank = {name: i for i, name in enumerate(JOURNEY_ORDER)}
intent_counts["stage_rank"] = intent_counts["journey_stage"].map(lambda s: stage_rank.get(s, 99))
intent_counts = intent_counts.sort_values(["stage_rank", "n", "intent"], ascending=[True, False, True])
intent_counts.to_csv(OUT_DIR / "A1_intent_counts.csv", index=False)

stage_counts = (
    intent_counts.groupby(["journey_stage", "risk_level"])["n"]
    .sum()
    .unstack(fill_value=0)
    .reindex(JOURNEY_ORDER, fill_value=0)
    .reindex(columns=["low", "medium", "high"], fill_value=0)
)
stage_share = stage_counts.div(stage_counts.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
stage_counts.assign(n=stage_counts.sum(axis=1)).to_csv(OUT_DIR / "A1_stage_risk.csv")
print((stage_share * 100).round(1).to_string())

plot_share = stage_share * 100
totals = stage_counts.sum(axis=1)
plot_share.index = [f"{name.replace('_', ' ')}    n={int(totals.loc[name]):,}" for name in plot_share.index]
fig, ax = plt.subplots(figsize=(10, 5.8))
plot_share.plot(
    kind="barh",
    stacked=True,
    color=[RISK_COLORS[c] for c in ["low", "medium", "high"]],
    ax=ax,
    width=0.72,
    legend=False,
)
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel("Share of sentences in the stage (%)")
ax.set_ylabel("")
handles = [plt.Rectangle((0, 0), 1, 1, color=RISK_COLORS[c]) for c in ["low", "medium", "high"]]
ax.legend(handles, ["low", "medium", "high"], title="Assigned risk", frameon=False, loc="lower left", bbox_to_anchor=(0, 1.02), ncol=3, borderaxespad=0)
fig.suptitle("A1. Assigned risk within each journey stage", fontweight="bold", x=0.01, ha="left", y=1.08)
fig.tight_layout()
save_fig(fig, "A_risk_by_stage")
plt.show()


risk_level          low  medium  high
journey_stage                        
card_setup         91.7     8.3   0.0
card_usage         79.5    20.5   0.0
payments           78.2    10.7  11.1
cash               82.8     0.0  17.2
top_up            100.0     0.0   0.0
transfers          63.4    36.6   0.0
limits            100.0     0.0   0.0
account_access     12.5     0.0  87.5
identity           25.5     0.0  74.5
foreign_exchange  100.0     0.0   0.0
refunds             0.0   100.0   0.0

**Interpretation.** Assigned risk belongs to the intent; the stage only summarizes those intents. Account access is 87.5% high-risk and identity is 74.5% high-risk, while top-up and foreign exchange are entirely low-risk. Transfers are 36.6% medium-risk and refunds are entirely medium-risk, so a stage that looks ordinary can still hold the intents a fraud desk would separate. These shares describe the Banking77 catalog, not confirmed fraudulent transactions.

## 2 A2. Which intents sit inside each stage?

The sunburst uses the same sentence counts as A1. Segment size is the number of catalog sentences. The inner ring is the journey stage, colored by its most common assigned risk, so a stage that is 87% high-risk is red even though it contains some low-risk intents. The outer ring is the intent, colored by that intent’s own risk. Only the 12 high-risk intents are named on the static figure. Every intent name, including the unlabeled ones, is in the hover text of `figures/A_sunburst.html`.

In [6]:
stage_of = (
    df.groupby("journey_stage")
    .agg(n=("text", "size"), top_risk=("risk_level", lambda s: s.value_counts().idxmax()))
    .reindex([s for s in JOURNEY_ORDER if s in set(df["journey_stage"])])
)

ids = ["all"]
labels = ["BANKING77"]
parents = [""]
values = [int(len(df))]
colors = ["#f4f6f7"]
custom = [["Banking77 catalog", f"{len(df):,} sentences, {df['intent'].nunique()} intents"]]
for stage, row in stage_of.iterrows():
    ids.append(stage)
    labels.append(stage.replace("_", " "))
    parents.append("all")
    values.append(int(row["n"]))
    colors.append(RISK_COLORS[row["top_risk"]])
    custom.append([stage.replace("_", " "), f"stage, most common risk: {row['top_risk']}"])
    sub = intent_counts[intent_counts["journey_stage"] == stage].sort_values("intent")
    for intent_row in sub.itertuples(index=False):
        ids.append(f"{stage}/{intent_row.intent}")
        short = {
            "card_payment_not_recognised": "unrecognised card payment",
            "cash_withdrawal_not_recognised": "unrecognised cash",
        }
        labels.append(short.get(intent_row.intent, intent_row.intent) if intent_row.intent in LABEL_INTENTS else "")
        parents.append(stage)
        values.append(int(intent_row.n))
        colors.append(RISK_COLORS[intent_row.risk_level])
        fraud_name = intent_row.fraud_type if pd.notna(intent_row.fraud_type) else "ordinary"
        custom.append([intent_row.intent, f"{intent_row.risk_level} risk, {fraud_name}"])

fig = go.Figure(go.Sunburst(
    ids=ids,
    labels=labels,
    parents=parents,
    values=values,
    customdata=custom,
    marker=dict(colors=colors, line=dict(color="white", width=1)),
    branchvalues="total",
    insidetextorientation="radial",
    hovertemplate="%{customdata[0]}<br>%{customdata[1]}<br>%{value} sentences<extra></extra>",
))
fig.update_layout(
    title=dict(
        text="A2. Journey stage and intent<br><sup>Inner ring: stage, colored by its most common assigned risk. Outer ring: intent, colored by its own assigned risk.</sup>",
        x=0.5,
        xanchor="center",
    ),
    margin=dict(t=80, l=16, r=16, b=78),
    width=1100,
    height=1100,
    paper_bgcolor="white",
    uniformtext=dict(minsize=9, mode="hide"),
)
fig.add_annotation(
    x=0.5, y=0.46, xref="paper", yref="paper", showarrow=False,
    text=f"{len(df):,} sentences<br>{df['intent'].nunique()} intents",
    font=dict(size=13, color="#34495e"),
)
legend = [("high, named", RISK_COLORS["high"]), ("medium", RISK_COLORS["medium"]), ("low", RISK_COLORS["low"])]
for i, (name, color) in enumerate(legend):
    fig.add_annotation(
        x=0.28 + i * 0.20, y=-0.045, xref="paper", yref="paper",
        text=f"● {name}", font=dict(color=color, size=14), showarrow=False,
    )
fig.write_image(FIG_DIR / "A_sunburst.png", scale=2)
fig.write_image(FIG_DIR / "A_sunburst.pdf")
fig.write_html(FIG_DIR / "A_sunburst.html", include_plotlyjs="cdn")
print(f"sunburst nodes={len(ids)} labeled high-risk intents={len(LABEL_INTENTS)}")


sunburst nodes=89 labeled high-risk intents=12

**Interpretation.** A blue stage is not a stage without high-risk intents: the inner color is only the most common assigned risk. The named outer slices are the 12 high-risk intents, and two of them, `card_payment_not_recognised` and `cash_withdrawal_not_recognised`, sit in payments and cash, whose stages are mostly ordinary. Segment size is the number of catalog sentences. The HTML figure names every unlabeled intent on hover.

## 3 A3. Do high-risk intents occupy their own region of meaning?

Each sentence is embedded with `all-MiniLM-L6-v2`. UMAP (`n_neighbors=15`, `min_dist=0.1`, cosine, `random_state=7`) is only the picture. The measurement is in the original embedding: 15 nearest neighbors by cosine distance. In scikit-learn 1.9, `kneighbors()` already omits the sentence itself, so the neighbor count is 15 rather than 16 followed by a slice.

Two rates are reported for each fraud type. The first is the share of those 15 neighbors with the same fraud type. The second uses only neighbors whose Banking77 intent differs from the query sentence, and asks what share of those still have the same fraud type. The second rate is the check that a tight cluster is not just many sentences of one intent.

Black marks are ordinary intents that most often appear beside account takeover or card compromise. Identity has almost no ordinary neighbors, so it is not marked. Labels on the map are display names; the tables keep the intent ids.

In [9]:
payload = (df["label"].astype(str) + "\t" + df["split"] + "\t" + df["text"]).tolist()
digest = hashlib.sha256("\n".join(payload).encode()).hexdigest()
meta = {"sha256": digest, "model": MODEL_NAME, "n": int(len(df)), "umap_random_state": 7}

def meta_matches():
    if not META_PATH.exists():
        return False
    saved = json.loads(META_PATH.read_text())
    return saved.get("sha256") == digest and saved.get("model") == MODEL_NAME and saved.get("n") == len(df)

if meta_matches() and EMB_PATH.exists():
    emb = np.load(EMB_PATH)
    print(f"loaded embeddings {emb.shape} from {EMB_PATH.name}")
else:
    from sentence_transformers import SentenceTransformer
    encoder = SentenceTransformer(MODEL_NAME)
    emb = encoder.encode(df["text"].tolist(), batch_size=64, normalize_embeddings=True, show_progress_bar=True).astype(np.float32)
    np.save(EMB_PATH, emb)
    print(f"saved embeddings {emb.shape}")

if meta_matches() and UMAP_PATH.exists():
    xy = np.load(UMAP_PATH)
    print(f"loaded UMAP coordinates from {UMAP_PATH.name}")
else:
    import umap
    xy = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=7, compatibility_layout=True).fit_transform(emb)
    xy = np.asarray(xy, dtype=np.float32)
    np.save(UMAP_PATH, xy)
    META_PATH.write_text(json.dumps(meta, indent=2))
    print("saved UMAP coordinates")
if not META_PATH.exists():
    META_PATH.write_text(json.dumps(meta, indent=2))

# sklearn 1.9 omits the query point when X is not passed, so n_neighbors=15 is 15 other sentences.
neighbors = NearestNeighbors(n_neighbors=15, metric="cosine").fit(emb)
idx = neighbors.kneighbors(return_distance=False)
high = df["risk_level"].eq("high").to_numpy()
base_high = float(high.mean())
df = df.copy()
df["neighbor_high_share"] = high[idx].mean(axis=1)

ordinary = df[df["fraud_type"].isna()]
mix = (
    ordinary.groupby("intent")
    .agg(n=("text", "size"), journey_stage=("journey_stage", "first"), risk_level=("risk_level", "first"), neighbor_high_share=("neighbor_high_share", "mean"))
    .reset_index()
    .sort_values("neighbor_high_share", ascending=False)
)
mix.to_csv(OUT_DIR / "A3_ordinary_overlap.csv", index=False)
print(f"base high-risk share={base_high:.3f}")
print(mix.head(8).round(3).to_string(index=False))

types = df["fraud_type_plot"].to_numpy()
intent_names = df["intent"].to_numpy()
touch_rows = []
for fraud_name in ["account_takeover", "card_compromise", "identity_kyc_bypass"]:
    mask = df["fraud_type"].eq(fraud_name).to_numpy()
    neigh_intents = intent_names[idx[mask]]
    neigh_types = types[idx[mask]]
    counts = pd.Series(neigh_intents[neigh_types == "(ordinary)"]).value_counts()
    total = int(mask.sum() * idx.shape[1])
    for intent_name, count in counts.head(3).items():
        touch_rows.append({
            "fraud_type": fraud_name,
            "ordinary_intent": intent_name,
            "neighbor_mentions": int(count),
            "share_of_neighborhood": count / total,
        })
touch = pd.DataFrame(touch_rows)
touch.to_csv(OUT_DIR / "A3_ordinary_neighbors_of_high_risk.csv", index=False)
print(touch.round(3).to_string(index=False))
marked_names = []
for fraud_name, group in touch.groupby("fraud_type", sort=False):
    group = group.sort_values("neighbor_mentions", ascending=False)
    if group.iloc[0]["neighbor_mentions"] >= 50:
        marked_names.append(group.iloc[0]["ordinary_intent"])
    if len(group) > 1 and group.iloc[1]["neighbor_mentions"] >= 100:
        marked_names.append(group.iloc[1]["ordinary_intent"])
marked = mix[mix["intent"].isin(marked_names)]

same_type = []
for name in FRAUD_COLORS:
    mask = types == name
    if name == "(ordinary)" or mask.sum() == 0:
        continue
    neigh_types = types[idx[mask]]
    neigh_intents = intent_names[idx[mask]]
    other = neigh_intents != intent_names[mask][:, None]
    same = neigh_types == name
    same_type.append({
        "fraud_type": name,
        "n": int(mask.sum()),
        "neighbor_same_type": float(same.mean()),
        "neighbor_ordinary": float((neigh_types == "(ordinary)").mean()),
        "share_neighbors_other_intent": float(other.mean()),
        "neighbor_same_type_excluding_same_intent": float((neigh_types[other] == name).mean()) if other.any() else np.nan,
    })
type_mix = pd.DataFrame(same_type).sort_values("neighbor_same_type", ascending=False)
type_mix.to_csv(OUT_DIR / "A3_fraud_type_neighbors.csv", index=False)
print(type_mix.round(3).to_string(index=False))

DISPLAY_INTENT = {
    "get_physical_card": "Physical card request",
    "pending_cash_withdrawal": "Pending cash withdrawal",
    "direct_debit_payment_not_recognised": "Unrecognized debit payment",
}
LABEL_OFFSET = {
    "get_physical_card": (-78, 18),
    "pending_cash_withdrawal": (12, 14),
    "direct_debit_payment_not_recognised": (12, -18),
}

fig, axes = plt.subplots(1, 2, figsize=(13.2, 6.6), gridspec_kw={"width_ratios": [1.35, 1], "wspace": 0.38})
ax = axes[0]
for name, color in FRAUD_COLORS.items():
    mask = df["fraud_type_plot"].eq(name).to_numpy()
    ax.scatter(
        xy[mask, 0], xy[mask, 1],
        s=5 if name == "(ordinary)" else 14,
        c=color,
        alpha=0.12 if name == "(ordinary)" else 0.85,
        label=FRAUD_LABELS[name],
        linewidths=0,
        zorder=1 if name == "(ordinary)" else 2,
    )
for row in marked.itertuples(index=False):
    mask = df["intent"].eq(row.intent).to_numpy()
    cx, cy = float(xy[mask, 0].mean()), float(xy[mask, 1].mean())
    ax.scatter([cx], [cy], s=36, c="black", marker="x", zorder=4)
    dx, dy = LABEL_OFFSET.get(row.intent, (8, 8))
    ax.annotate(
        DISPLAY_INTENT.get(row.intent, row.intent),
        (cx, cy),
        textcoords="offset points",
        xytext=(dx, dy),
        fontsize=8,
        color="#1c2833",
        arrowprops=dict(arrowstyle="-", color="#1c2833", lw=0.6),
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.92, pad=1.2),
        zorder=5,
    )
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.set_title("Where the sentences sit", loc="left")
ax.legend(frameon=False, markerscale=1.8, loc="lower left", fontsize=7.5, title="Intent-level fraud type")

ax = axes[1]
order = type_mix.sort_values("neighbor_same_type", ascending=True)
y = np.arange(len(order))
ax.barh(y + 0.16, order["neighbor_same_type"] * 100, height=0.32, color="#1c2833", label="Same fraud type, all 15 neighbors")
ax.barh(y - 0.16, order["neighbor_same_type_excluding_same_intent"] * 100, height=0.32, color="#85929e", label="Same fraud type, different intent only")
ax.set_yticks(y, [FRAUD_LABELS[name] for name in order["fraud_type"]])
ax.set_xlim(0, 100)
ax.set_xlabel("Percent of neighbors")
ax.set_title("Separation in the embedding", loc="left")
ax.legend(frameon=False, fontsize=7.5, loc="upper center", bbox_to_anchor=(0.5, -0.18), borderaxespad=0)
fig.suptitle("A3. Semantic similarity across fraud categories", fontweight="bold", x=0.01, ha="left")
fig.text(0.01, 0.01, "UMAP of MiniLM embeddings. Color is the intent-level fraud type, not a confirmed fraud label.", fontsize=8, color="#5d6d7e")
fig.subplots_adjust(bottom=0.24, top=0.86)
save_fig(fig, "A_umap")
plt.show()


loaded embeddings (13083, 384) from A_minilm_embeddings.npy
loaded UMAP coordinates from A_umap_xy.npy
base high-risk share=0.144
                             intent   n journey_stage risk_level  neighbor_high_share
direct_debit_payment_not_recognised 222      payments        low                0.102
                  get_physical_card 146    card_setup        low                0.084
      wrong_amount_of_cash_received 220          cash        low                0.045
            pending_cash_withdrawal 183          cash        low                0.041
                   card_not_working 152    card_usage        low                0.040
                     card_swallowed 101    card_usage        low                0.040
                       card_arrival 193    card_setup        low                0.029
                       card_linking 179    card_setup        low                0.028
         fraud_type                     ordinary_intent  neighbor_mentions  share_of_neighborhoo

**Interpretation.** Identity wording coheres across intents: 98.5% of neighbors share the fraud type, and 90.6% of the neighbors from a different intent are still identity. Account takeover looks equally tight at 95.5%, but only 6.1% of its neighbors leave the original intent, and 26.3% of those are still takeover, so the cluster is mostly one intent repeated. Card compromise is the overlap a queue would feel: 12.1% of its neighbors are ordinary, led by unrecognized debit payments and pending cash withdrawals, while those ordinary intents themselves have only 10.2% and 4.1% high-risk neighbors, below the 14.4% base rate.

## 4 A4. Does sentence length differ by assigned risk?

Length is kept separate from the stage chart so the 77 intent names do not stretch the distribution. The axis stops at 200 characters. The catalog has 186 sentences longer than that, almost all of them medium-risk refund and transfer wording. The table under the chart is the summary; the drawing shows the shape.

In [12]:
length_by_risk = (
    df.groupby("risk_level")["text_length"]
    .agg(n="size", median="median", mean="mean", p10=lambda s: s.quantile(0.10), p90=lambda s: s.quantile(0.90))
    .reindex(["low", "medium", "high"])
    .round(1)
)
length_by_risk.to_csv(OUT_DIR / "A1_length_by_risk.csv")
print(length_by_risk)
print(f"sentences longer than 200 characters: {int((df['text_length'] > 200).sum())}")

order = ["low", "medium", "high"]
samples = [df.loc[df["risk_level"] == risk, "text_length"].to_numpy() for risk in order]
fig = plt.figure(figsize=(7.2, 6.0))
ax = fig.add_axes([0.10, 0.34, 0.86, 0.56])
parts = ax.violinplot(samples, positions=[1, 2, 3], showmedians=False, showextrema=False)
for body, risk in zip(parts["bodies"], order):
    body.set_facecolor(RISK_COLORS[risk])
    body.set_alpha(0.85)
    body.set_edgecolor("white")
ax.boxplot(
    samples,
    positions=[1, 2, 3],
    widths=0.12,
    showfliers=False,
    patch_artist=True,
    medianprops={"color": "#1c2833", "linewidth": 1.4},
    boxprops={"facecolor": "white", "edgecolor": "#1c2833", "linewidth": 0.8},
    whiskerprops={"color": "#1c2833", "linewidth": 0.8},
    capprops={"color": "#1c2833", "linewidth": 0.8},
)
ax.set_xticks([1, 2, 3], order)
ax.set_xlim(0.4, 3.6)
ax.set_ylim(0, 200)
ax.set_ylabel("Characters")
ax.set_xlabel("Assigned risk")
ax.set_title("A4. Sentence length by assigned risk", loc="left", fontweight="bold")
ax_table = fig.add_axes([0.18, 0.08, 0.72, 0.16])
ax_table.axis("off")
table = ax_table.table(
    cellText=[["9,466", "46", "99"], ["1,730", "50", "142"], ["1,887", "45", "90"]],
    rowLabels=["low", "medium", "high"],
    colLabels=["n", "median", "p90"],
    cellLoc="center",
    loc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.25)
for cell in table.get_celld().values():
    cell.set_edgecolor("#d5d8dc")
    cell.set_linewidth(0.4)
fig.text(0.10, 0.02, "Axis stops at 200 characters. 186 sentences are longer.", fontsize=8, color="#5d6d7e")
save_fig(fig, "A_length_by_risk")
plt.show()


n  median  mean   p10    p90
risk_level                                 
low         9466    46.0  57.3  29.0   99.0
medium      1730    50.0  67.2  31.0  142.0
high        1887    45.0  54.1  27.0   90.0
sentences longer than 200 characters: 186

**Interpretation.** Message lengths are broadly similar across assigned risk levels: the median is 46 characters for low, 50 for medium, and 45 for high. Medium-risk sentences have the longer tail, with a 90th percentile of 142 characters against 90 for high-risk. Length alone may have limited value for distinguishing these assigned risk levels. This is a description of the catalog, not a test of predictive performance.

## 5 Summary for the progress report

1.  Assigned risk is an intent property. Account access and identity are mostly high-risk intents. `card_payment_not_recognised` and `cash_withdrawal_not_recognised` are high-risk intents inside stages whose most common risk is low. A stage label and an intent label are not the same feature.
2.  Sentence length is broadly similar across assigned risk levels. It is a weak descriptive separator, and this notebook does not test it as a predictor.
3.  In the embedding, identity language coheres across different intents. Account-takeover separation is mostly sentences of the same intent sitting together. Card-compromise wording meets ordinary complaints that a payment or a cash withdrawal was not recognised.
4.  All of these labels come from `dim_intent` joined onto Banking77. They say which kinds of wording are separable. They do not say how accurately a later system will detect fraudulent conversations.